# Day 2 — Retrieval Optimization
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 2 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

Day 1 built an index that returns *something*. Today is about proving it returns the
*right* thing — with real, measured numbers instead of a single example query.

**By the end of this notebook you will be able to:**
1. Explain how `top_k` trades off precision against coverage
2. Run a controlled experiment comparing chunk-size configurations
3. Build a small test set and compute Retrieval Precision@k by hand
4. Read your own results and decide what to change before Day 3

> This notebook rebuilds the Day 1 index at the top so it's self-contained — you can run
> it independently without re-running `Day1_Document_Ingestion.ipynb` first.


## 0. Setup — Rebuild the Day 1 Index


In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src import config
from src.data_loader import load_pdfs, chunk_documents
from src.llm_engine import build_index
def retrieve(vectorstore, query, k=3):
    return vectorstore.similarity_search_with_relevance_scores(query, k=k)

pages = load_pdfs()
chunks = chunk_documents(pages)
vectordb = build_index(chunks)
print(f"\nIndex ready: {len(chunks)} chunks from {len(pages)} pages.")


c:\Users\ABDELRAHMAN GAMAL\Desktop\CDSS\src\llm_engine.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name='NeuML/pubmedbert-base-embeddings')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Index ready: 1686 chunks from 333 pages.


## 1. What `top_k` Actually Controls

`top_k` is the number of chunks retrieval hands to the generation step. It is not a minor
setting — it's your first real trade-off of the day:

| `k` | Effect | Risk |
|---|---|---|
| Too low (1–2) | Very focused | Misses relevant evidence sitting in another section |
| Balanced (3–5) | Good coverage, manageable context | Usually the right starting point |
| Too high (10+) | Broad coverage | Dilutes context, invites irrelevant or contradictory chunks |

Let's see this directly: run the same question at three different `k` values and compare.


In [2]:
question = "What are the contraindications for treatment with PD-1/PD-L1 inhibitors?"

for k in [1, 3, 8]:
    results = retrieve(vectordb, question, k=k)
    print(f"--- k={k} ---")
    for doc, score in results:
        print(f"  score={score:.3f}  page {doc.metadata.get('page_number')}: "
              f"{doc.page_content[:70].strip()}...")
    print()


--- k=1 ---
  score=-68.672  page 150: antitumor immunity; PD-1 receptors are expressed on activated cytotoxi...

--- k=3 ---
  score=-68.672  page 150: antitumor immunity; PD-1 receptors are expressed on activated cytotoxi...
  score=-68.672  page 150: antitumor immunity; PD-1 receptors are expressed on activated cytotoxi...
  score=-76.384  page 105: NCCN Guidelines Version 7.2026
Non-Small Cell Lung Cancer
Version 7.20...

--- k=8 ---
  score=-68.672  page 150: antitumor immunity; PD-1 receptors are expressed on activated cytotoxi...
  score=-68.672  page 150: antitumor immunity; PD-1 receptors are expressed on activated cytotoxi...
  score=-76.384  page 105: NCCN Guidelines Version 7.2026
Non-Small Cell Lung Cancer
Version 7.20...
  score=-76.384  page 105: NCCN Guidelines Version 7.2026
Non-Small Cell Lung Cancer
Version 7.20...
  score=-76.676  page 172: except for EGFR exon 20 mutations or KRAS mutations. Continuation 
mai...
  score=-76.676  page 172: except for EGFR exon 20 mu

C:\Users\ABDELRAHMAN GAMAL\AppData\Local\Temp\ipykernel_5632\2666816527.py:8: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'page_number': 150, 'moddate': '2026-08-17T17:55:15-04:00', 'author': '', 'document_name': 'nscl.pdf', 'producer': 'PDFsharp 6.2.1 (Original: Adobe PDF Library 18.0)', 'keywords': '', 'page': 149, 'format': 'PDF 1.7', 'creationDate': "D:20260807080739-04'00'", 'title': '', 'creationdate': '2026-08-07T08:07:39-04:00', 'total_pages': 302, 'source': 'C:\\Users\\ABDELRAHMAN GAMAL\\Desktop\\CDSS\\data\\raw\\nscl.pdf', 'subject': '', 'file_path': 'C:\\Users\\ABDELRAHMAN GAMAL\\Desktop\\CDSS\\data\\raw\\nscl.pdf', 'creator': 'Adobe InDesign 21.4 (Windows)', 'trapped': '', 'encryption': 'Standard V2 R3 128-bit RC4', 'modDate': "D:20260817175515-04'00'"}, page_content='antitumor immunity; PD-1 receptors are expressed on activated cytotoxic \nT cells.347-349 Cemiplimab-rwlc, nivolumab, and pembrolizumab inhibit \nPD-1 receptors.119,350,351 

### Checkpoint 1

Look at the `k=8` output. Are all 8 results still genuinely about target blood pressure, or
do the later ones start drifting into unrelated sections of the guideline? This drift —
not an error, just noise — is exactly why `top_k` needs to be tuned deliberately rather
than set high "to be safe."


## 2. Ablation Experiment — Chunk Size & Overlap

A proper experiment needs a fixed method: same source, same queries, only the chunking
configuration changes. We'll rebuild the index three times with different `chunk_size` /
`chunk_overlap` pairs and compare retrieval on a fixed query set.


In [3]:
import importlib
from langchain_text_splitters import RecursiveCharacterTextSplitter
from src.llm_engine import get_embedding_function
from langchain_community.vectorstores import Chroma

test_queries = [
    "What blood pressure threshold should trigger starting medication?",
    "What are the three recommended first-line drug classes?",
    "Can nurses or pharmacists prescribe antihypertensive treatment?",
]

configurations = [
    {"name": "Small (200/0)",    "chunk_size": 200,  "chunk_overlap": 0},
    {"name": "Balanced (400/50)", "chunk_size": 400,  "chunk_overlap": 50},
    {"name": "Large (600/100)",  "chunk_size": 600,  "chunk_overlap": 100},
]

embed_fn = get_embedding_function()
experiment_results = []

for cfg in configurations:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"] * 4,
        chunk_overlap=cfg["chunk_overlap"] * 4,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    test_chunks = splitter.split_documents(pages)
    test_db = Chroma.from_documents(
        documents=test_chunks, embedding=embed_fn,
        collection_name=f"experiment_{cfg['chunk_size']}",
    )

    avg_score = 0
    for q in test_queries:
        results = test_db.similarity_search_with_relevance_scores(q, k=3)
        avg_score += sum(s for _, s in results) / len(results)
    avg_score /= len(test_queries)

    experiment_results.append({"config": cfg["name"], "n_chunks": len(test_chunks), "avg_top3_score": avg_score})
    print(f"{cfg['name']:<20} chunks={len(test_chunks):>4}   avg top-3 relevance={avg_score:.3f}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\ABDELRAHMAN GAMAL\AppData\Local\Temp\ipykernel_5632\3445166501.py:35: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'format': 'PDF 1.7', 'author': '', 'creationDate': "D:20260807080739-04'00'", 'document_name': 'nscl.pdf', 'page_number': 48, 'title': '', 'subject': '', 'moddate': '2026-08-17T17:55:15-04:00', 'creationdate': '2026-08-07T08:07:39-04:00', 'page': 47, 'creator': 'Adobe InDesign 21.4 (Windows)', 'source': 'C:\\Users\\ABDELRAHMAN GAMAL\\Desktop\\CDSS\\data\\raw\\nscl.pdf', 'producer': 'PDFsharp 6.2.1 (Original: Adobe PDF Library 18.0)', 'modDate': "D:20260817175515-04'00'", 'file_path': 'C:\\Users\\ABDELRAHMAN GAMAL\\Desktop\\CDSS\\data\\raw\\nscl.pdf', 'trapped': '', 'total_pages': 302, 'keywords': '', 'encryption': 'Standard V2 R3 128-bit RC4'}, page_content='venous thromboembolic events. \nyy Prophylaxis with oral doxycycline or minocycline, clindamycin lotion applied \nto the scalp, chlorhexidine applied to nails, and a ceramide

Small (200/0)        chunks=1850   avg top-3 relevance=-156.558


KeyboardInterrupt: 

### Checkpoint 2

This is a real experiment, not a demonstration — the numbers above come from your actual
index, actual embedding model, and actual test queries. Before moving on:

- Which configuration scored highest on average?
- Did the configuration with the *most* chunks also score the *best*? (It often doesn't —
  more chunks means more noise to filter through, not automatically better retrieval.)

Record which configuration you're keeping in `config.py` and why — you'll want that
justification ready when a judge asks about it on Day 5.


## 3. Build Your Test Set

A single example query proves nothing. `eval/Day2_Evaluation_Test_Set.csv` — already in
your starter kit — has 8 real questions with verified expected sources. Let's load it and
use it to compute a real metric.


In [9]:
import csv

test_set = []
with open("eval/Day2_Evaluation_Test_Set.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        test_set.append(row)

print(f"Loaded {len(test_set)} test questions.\n")
for row in test_set[:3]:
    print("Q:", row["Question"])
    print("   Expected:", row["Expected Source (Document / Section / Page)"])
    print()


Loaded 8 test questions.

Q: What blood pressure threshold should trigger starting medication for hypertension?
   Expected: WHO Hypertension Guideline / Recommendation 1: Blood pressure threshold for initiation of pharmacological treatment / Page 3

Q: What are the three recommended first-line drug classes for treating hypertension?
   Expected: WHO Hypertension Guideline / 3.4 Drug classes to be used as first-line agents / Page 8

Q: What is the target blood pressure for a patient with existing cardiovascular disease?
   Expected: WHO Hypertension Guideline / 3.6 Target blood pressure / Page 9



## 4. Compute Retrieval Precision@k

$$\text{Precision@k} = \frac{\text{relevant chunks in top-k}}{k}$$

For each question, we check whether the retrieved chunks' page numbers match the expected
page from the test set. This is a simplified, page-level version of Precision@k — good
enough to get a real, defensible number today.


In [10]:
def page_matches_expected(retrieved_page, expected_text):
    """Very simple check: does the expected 'Page N' appear, and does our retrieved
    page number match it? Returns None for the deliberate out-of-scope question."""
    import re
    m = re.search(r"Page (\d+)", expected_text)
    if not m:
        return None  # out-of-scope control question — handled separately
    expected_page = int(m.group(1))
    return retrieved_page == expected_page


k = 3
precisions = []
print(f"{'Question':<55} {'P@' + str(k):<8} Notes")
print("-" * 90)

for row in test_set:
    expected = row["Expected Source (Document / Section / Page)"]
    if "Not covered" in expected:
        print(f"{row['Question'][:53]:<55} {'N/A':<8} out-of-scope control question")
        continue

    results = retrieve(vectordb, row["Question"], k=k)
    hits = sum(
        1 for doc, _ in results
        if page_matches_expected(doc.metadata.get("page_number"), expected)
    )
    precision = hits / k
    precisions.append(precision)
    print(f"{row['Question'][:53]:<55} {precision:<8.2f}")

avg_precision = sum(precisions) / len(precisions)
print("-" * 90)
print(f"Average Precision@{k} across {len(precisions)} scored questions: {avg_precision:.2f}")


Question                                                P@3      Notes
------------------------------------------------------------------------------------------
What blood pressure threshold should trigger starting   0.00    
What are the three recommended first-line drug classe   0.00    
What is the target blood pressure for a patient with    0.00    
How often should a patient be followed up after start   0.00    


C:\Users\ABDELRAHMAN GAMAL\AppData\Local\Temp\ipykernel_16344\2666816527.py:8: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'file_path': 'data\\raw\\chapter 5 (1).pdf', 'keywords': '', 'creationdate': '', 'total_pages': 10, 'page_number': 7, 'title': '', 'moddate': '2026-08-17T11:34:55+00:00', 'producer': 'iLovePDF', 'subject': '', 'page': 6, 'modDate': 'D:20260817113455Z', 'document_name': 'chapter 5 (1).pdf', 'trapped': '', 'author': '', 'creator': '', 'source': 'data\\raw\\chapter 5 (1).pdf', 'creationDate': '', 'format': 'PDF 1.5'}, page_content='against a standard treatment. If the \nresults are good, it may be approved by \nthe FDA. \n> Phase 4 trials study the safety and \nbenefit of an FDA-approved treatment. \nWho can enroll? \nIt depends on the clinical trial’s rules, called \neligibility criteria. The rules may be about age, \ncancer type and stage, treatment history, or'), -185.59794410322587), (Document(metadata={'page_number': 7, 'format

Can nurses or pharmacists prescribe antihypertensive    0.00    
Which antihypertensive medications are contraindicate   0.00    
What is the recommended starting dose in the single-p   0.00    
What screening interval does this guideline recommend   N/A      out-of-scope control question
------------------------------------------------------------------------------------------
Average Precision@3 across 7 scored questions: 0.00


C:\Users\ABDELRAHMAN GAMAL\AppData\Local\Temp\ipykernel_16344\2666816527.py:8: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'creator': '', 'page': 11, 'producer': 'iLovePDF', 'creationdate': '', 'modDate': 'D:20260817113141Z', 'subject': '', 'author': '', 'document_name': 'lung cancer.pdf', 'moddate': '2026-08-17T11:31:41+00:00', 'page_number': 12, 'total_pages': 21, 'keywords': '', 'creationDate': '', 'file_path': 'C:\\Users\\ABDELRAHMAN GAMAL\\Desktop\\CDSS\\data\\raw\\lung cancer.pdf', 'format': 'PDF 1.5', 'title': '', 'source': 'C:\\Users\\ABDELRAHMAN GAMAL\\Desktop\\CDSS\\data\\raw\\lung cancer.pdf', 'trapped': ''}, page_content='» Hyponatremia, euvolemia, low serum osmolality, inappropriately concentrated urine osmolality, normal thyroid and adrenal function \n« Cushing syndrome?: \n» Ectopic adrenocorticotropic hormone (ACTH) secretion \n» Weight gain, moon facies, hypertension, hyperglycemia, generalized weakness \n» High serum cortisol and AC

### Checkpoint 3 — Day 2 Self-Check

- [ ] You ran the same query at 3 different `k` values and can explain the trade-off out loud
- [ ] You ran a real ablation experiment across 3 chunking configurations and picked one
- [ ] You have an actual Precision@k number — not a guess — for your current index
- [ ] `config.py` reflects the configuration you're keeping, and you know why

For extra rigor, open `templates/Day2_Retrieval_Scorecard_Template.xlsx` and log these
same numbers there — it has live formulas so your team average recalculates automatically
as more teammates fill it in.

## What's Next

Day 3's notebook assumes retrieval is now trustworthy — it moves on to constraining the
model so tightly that every generated answer can only say what these retrieved chunks
actually support.
